In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama 서버 시작됨")

In [ ]:
!ollama pull exaone3.5:7.8b

In [ ]:
!ollama list

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
print("exaone_generated 폴더:", glob.glob("/content/drive/MyDrive/FinHOLLY/exaone_generated/*"))

In [ ]:
import os
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

try:
    resp = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": "Translate to Vietnamese: 안녕하세요"}],
        max_tokens=50,
    )
    print("성공:", resp.choices[0].message.content)
except Exception as e:
    print("실패:", repr(e))

In [ ]:
import os
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

models = client.models.list()
for m in models.data:
    print(m.id)

In [ ]:
import subprocess, json, re, os, glob
import pandas as pd

def ask_exaone(prompt):
    r = subprocess.run(["ollama", "run", "exaone3.5:7.8b", prompt],
                       capture_output=True, text=True, timeout=300)
    return r.stdout.strip()

terms = {
    "차주": ("금융: 대출자(借主)", "일상: 다음 주(次週)"),
    "청산": ("금융: 채권채무·회사 정리(淸算)", "일상: 관계·과거 정리"),
    "사모": ("금융: 소수 투자자 대상(私募)", "일상: 그리워함(思慕)"),
    "모집": ("금융: 증권 공모·자금(募集)", "일상: 사람·회원 모음"),
}

results = {}
for term, (fin, daily) in terms.items():
    print(f"=== {term} 생성 중 ===")
    p = f"""'{term}'은 두 가지 뜻이 있습니다:
- {fin}
- {daily}
각 뜻으로 자연스러운 한국어 문장을 5개씩 만들어줘. 번호·별표 없이 이 형식만:
[금융] 문장
[금융] 문장
[금융] 문장
[금융] 문장
[금융] 문장
[일상] 문장
[일상] 문장
[일상] 문장
[일상] 문장
[일상] 문장"""
    results[term] = ask_exaone(p)
    print(results[term], "\n")

def parse(text, term):
    rows, label = [], None
    for line in text.split("\n"):
        line = line.strip()
        if not line: continue
        if "[금융]" in line: label="pos"; line=line.replace("[금융]","").strip()
        elif "[일상]" in line: label="neg"; line=line.replace("[일상]","").strip()
        line = re.sub(r'^\*+\s*','',line); line=re.sub(r'\*\*','',line)
        line = re.sub(r'^\d+\.\s*','',line).strip()
        if line and line!="." and label and len(line)>5:
            rows.append({"term":term,"label":label,"ko_sentence":line})
    return rows

all_rows = []
for t in terms: all_rows.extend(parse(results[t], t))
df = pd.DataFrame(all_rows)

SAVE_DIR = "/content/drive/MyDrive/FinHOLLY/exaone_generated"
os.makedirs(SAVE_DIR, exist_ok=True)
raw_path = f"{SAVE_DIR}/raw_sentences.json"
clean_path = f"{SAVE_DIR}/cleaned_sentences.csv"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
df.to_csv(clean_path, index=False, encoding="utf-8-sig")

print("=== 저장 검증 ===")
found = glob.glob(f"{SAVE_DIR}/*")
print("폴더 내용:", found)
if os.path.exists(clean_path):
    reread = pd.read_csv(clean_path, encoding="utf-8-sig")
    print(f"✅ cleaned 저장 성공: {len(reread)}행")
    for t in terms:
        sub = reread[reread["term"]==t]
        print(f"  {t}: 금융 {len(sub[sub['label']=='pos'])} / 일상 {len(sub[sub['label']=='neg'])}")
else:
    print("❌ 저장 실패")

In [ ]:
import os, time, glob
import pandas as pd
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ["GROQ_API_KEY"])

SAVE_DIR = "/content/drive/MyDrive/FinHOLLY/exaone_generated"
df = pd.read_csv(f"{SAVE_DIR}/cleaned_sentences.csv", encoding="utf-8-sig")
print(f"번역할 문장: {len(df)}개")

LANGS = {
    "vie_Latn": "Vietnamese", "ind_Latn": "Indonesian", "tha_Thai": "Thai",
    "tgl_Latn": "Filipino (Tagalog)", "mya_Mymr": "Burmese (Myanmar)",
}

def translate(ko_text, lang_name, sense_hint):
    prompt = f"""Translate this Korean financial-domain sentence into {lang_name}.
Context: the key term is used in a {sense_hint} sense.
Output ONLY the translation, no explanation, no quotes.

Korean: {ko_text}"""
    try:
        resp = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3, max_tokens=256,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[ERROR: {e}]"

rows = []
total = len(df) * len(LANGS); done = 0
for _, r in df.iterrows():
    sense = "financial" if r["label"] == "pos" else "everyday/non-financial"
    for code, name in LANGS.items():
        tr = translate(r["ko_sentence"], name, sense)
        rows.append({"term": r["term"], "label": r["label"],
                     "ko_sentence": r["ko_sentence"], "lang": code, "translation": tr})
        done += 1
        if done % 20 == 0: print(f"  {done}/{total}")
        time.sleep(0.5)

out_df = pd.DataFrame(rows)
out_path = f"{SAVE_DIR}/translated_pairs.csv"
out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\n=== 저장 검증 ===")
print("파일:", glob.glob(out_path))
if os.path.exists(out_path):
    print(f"✅ 번역 저장 성공: {len(pd.read_csv(out_path, encoding='utf-8-sig'))}행")

print("\n=== 샘플 ===")
for code in LANGS:
    s = out_df[out_df["lang"] == code].iloc[0]
    print(f"[{code}] {s['term']}({s['label']}): {s['translation']}")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/FinHOLLY/exaone_generated"
df = pd.read_csv(f"{SAVE_DIR}/translated_pairs.csv", encoding="utf-8-sig")
print(f"검사할 번역: {len(df)}행")

df = df[~df["translation"].astype(str).str.startswith("[ERROR", na=False)].copy()
df = df.dropna(subset=["translation"])
df = df[df["translation"].astype(str).str.strip() != ""].copy()
df["translation"] = df["translation"].astype(str)
print(f"정제 후: {len(df)}행")

results = []
for (term, lang), grp in df.groupby(["term", "lang"]):
    pos = [t for t in grp[grp["label"]=="pos"]["translation"].tolist() if t.strip()]
    neg = [t for t in grp[grp["label"]=="neg"]["translation"].tolist() if t.strip()]
    if not pos or not neg:
        continue
    pos_emb = labse.encode(pos, normalize_embeddings=True).mean(axis=0)
    neg_emb = labse.encode(neg, normalize_embeddings=True).mean(axis=0)
    sim = float(np.dot(pos_emb, neg_emb))
    results.append({"term": term, "lang": lang, "pos_neg_sim": round(sim,3), "margin": round(1-sim,3)})

res_df = pd.DataFrame(results).sort_values("margin", ascending=False)
print("\n=== 용어×언어별 대조 효과 (margin 클수록 잘 갈림) ===")
print(res_df.to_string())

THRESHOLD = 0.05
passed = res_df[res_df["margin"] >= THRESHOLD]
failed = res_df[res_df["margin"] < THRESHOLD]
print(f"\n통과(margin>={THRESHOLD}): {len(passed)} / 전체 {len(res_df)}")
if len(failed) > 0:
    print("\n탈락:")
    print(failed.to_string())

res_df.to_csv(f"{SAVE_DIR}/labse_filter_report.csv", index=False, encoding="utf-8-sig")
print(f"\n리포트 저장 완료")

In [ ]:
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/FinHOLLY/data"
SAVE_DIR = "/content/drive/MyDrive/FinHOLLY/exaone_generated"

new_df = pd.read_csv(f"{SAVE_DIR}/translated_pairs.csv", encoding="utf-8-sig")
new_df = new_df.dropna(subset=["translation"])
new_df = new_df[new_df["translation"].astype(str).str.strip() != ""].copy()

train = pd.read_csv(f"{DATA_DIR}/contrastive_train.csv", encoding="utf-8-sig")
val = pd.read_csv(f"{DATA_DIR}/contrastive_val.csv", encoding="utf-8-sig")
test = pd.read_csv(f"{DATA_DIR}/contrastive_test.csv", encoding="utf-8-sig")

print("=== 기존 split의 용어 ===")
print("train 용어:", sorted(train["term"].unique()))
print("val 용어:", sorted(val["term"].unique()))
print("test 용어:", sorted(test["term"].unique()))
print("\n=== 새 데이터 용어 ===")
print("new 용어:", sorted(new_df["term"].unique()))
print(f"new 행수: {len(new_df)}")

for term in sorted(new_df["term"].unique()):
    where = []
    if term in train["term"].values: where.append("train")
    if term in val["term"].values: where.append("val")
    if term in test["term"].values: where.append("test")
    print(f"  {term}: 기존 {where if where else '없음(신규)'}")

In [ ]:
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/FinHOLLY/data"
SAVE_DIR = "/content/drive/MyDrive/FinHOLLY/exaone_generated"

new_df = pd.read_csv(f"{SAVE_DIR}/translated_pairs.csv", encoding="utf-8-sig")
new_df = new_df.dropna(subset=["translation"])
new_df = new_df[new_df["translation"].astype(str).str.strip() != ""].copy()

train = pd.read_csv(f"{DATA_DIR}/contrastive_train.csv", encoding="utf-8-sig")
val = pd.read_csv(f"{DATA_DIR}/contrastive_val.csv", encoding="utf-8-sig")
test = pd.read_csv(f"{DATA_DIR}/contrastive_test.csv", encoding="utf-8-sig")

print("기존 컬럼:", list(val.columns))
print("새 데이터 컬럼:", list(new_df.columns))

common_cols = [c for c in val.columns if c in new_df.columns]
print("공통 컬럼:", common_cols)

val_terms = ["차주"]
test_terms = ["모집", "사모", "청산"]

new_to_val = new_df[new_df["term"].isin(val_terms)][common_cols]
new_to_test = new_df[new_df["term"].isin(test_terms)][common_cols]

val_merged = pd.concat([val[common_cols], new_to_val], ignore_index=True)
test_merged = pd.concat([test[common_cols], new_to_test], ignore_index=True)

print("\n=== 병합 결과 ===")
print(f"train: {len(train)}행 (변경 없음)")
print(f"val:  {len(val)} → {len(val_merged)}행 (+{len(new_to_val)})")
print(f"test: {len(test)} → {len(test_merged)}행 (+{len(new_to_test)})")

overlap_val = set(train["term"]) & set(val_merged["term"])
overlap_test = set(train["term"]) & set(test_merged["term"])
overlap_vt = set(val_merged["term"]) & set(test_merged["term"])
print(f"\n=== term 분리 검증 ===")
print(f"train∩val: {overlap_val if overlap_val else '없음 ✅'}")
print(f"train∩test: {overlap_test if overlap_test else '없음 ✅'}")
print(f"val∩test: {overlap_vt if overlap_vt else '없음 ✅'}")

val_merged.to_csv(f"{DATA_DIR}/contrastive_val_expanded.csv", index=False, encoding="utf-8-sig")
test_merged.to_csv(f"{DATA_DIR}/contrastive_test_expanded.csv", index=False, encoding="utf-8-sig")
print(f"\n저장 완료:")
print(f"  contrastive_val_expanded.csv ({len(val_merged)}행)")
print(f"  contrastive_test_expanded.csv ({len(test_merged)}행)")